<a href="https://colab.research.google.com/github/Abdoulaye279/Financial-Engineering/blob/main/var-es-sp500/05_risk_weights.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 05 - Diversification and risk weights

How much the 50/50 portfolio really diversifies, where its tail risk comes from, and the weights that split the ES equally.

In [1]:
# colab: get the repo first
try:
    import google.colab
    !git clone -q https://github.com/Abdoulaye279/Financial-Engineering
    %cd Financial-Engineering/var-es-sp500
except ImportError:
    pass

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from risk import *
plt.style.use("quant.mplstyle")

prices = pd.concat({"spx": load_close("../black-scholes-sp500/data/sp500.csv"),
                    "aapl": load_close("data/aapl.csv")}, axis=1).dropna()
rets = np.log(prices).diff().dropna()
simple = np.exp(rets) - 1
w = np.array([0.5, 0.5])
port = np.log1p(simple @ w)   # 50/50, rebalanced every day
len(rets), rets.index[0].date(), rets.index[-1].date()

(6761, datetime.date(2000, 1, 4), datetime.date(2026, 11, 19))

In [3]:
var_s, _ = var_es_hist(rets.spx.values)
var_a, _ = var_es_hist(rets.aapl.values)
var_p, _ = var_es_hist(port.values)
print(f"historical VaR 99%: spx {var_s:.2%}  aapl {var_a:.2%}  portfolio {var_p:.2%}")
print(f"gain vs holding the two risks separately: {1 - var_p / (0.5 * var_s + 0.5 * var_a):.1%}")
# only -13%: on the worst days they tend to fall together (see 03)

historical VaR 99%: spx 3.46%  aapl 6.37%  portfolio 4.25%
gain vs holding the two risks separately: 13.4%


In [4]:
print("share of the ES, 50/50:", es_contributions(simple.values, w).round(3))
print("since 2010:            ", es_contributions(simple.loc["2010":].values, w).round(3))
# apple brings 68% of the ES with 50% of the money (60% since 2010)

share of the ES, 50/50: [0.316 0.684]
since 2010:             [0.399 0.601]


In [5]:
w_eq = equal_risk_weights(simple.values)
print("equal ES weights:", w_eq.round(3), " check:", es_contributions(simple.values, w_eq).round(3))
for name, ww in [("50/50", w), ("equal ES", w_eq)]:
    p = np.log1p(simple.values @ ww)
    v, e = var_es_hist(p)
    print(f"{name:9s} return {252 * p.mean():.1%}  vol {np.sqrt(252) * p.std():.1%}  VaR {v:.2%}  ES {e:.2%}")
# apple only 36% for an equal split of the ES. ES 6.27% -> 5.68%
# but the return drops too (15.6% -> 13.1%) since apple did very well: less dependence on one stock, not a free lunch
# careful: weights computed on the whole period = look-ahead. a real version would recompute them every year

equal ES weights: [0.641 0.359]  check: [0.5 0.5]
50/50     return 15.6%  vol 26.0%  VaR 4.25%  ES 6.27%
equal ES  return 13.1%  vol 23.2%  VaR 3.90%  ES 5.68%
